# PI3 Grupo 4 — Sprint 2 | Pipeline piloto de extração radiômica (v2)

**Responsáveis:** Rafael e Samara · **Etapa KDD:** Transformação

**Pergunta da sprint:** as segmentações preparadas permitem extrair de forma
consistente features de forma, intensidade e textura?

---

### O que mudou desde a v1

A v1 rodou duas vezes sobre os mesmos 25 pacientes e produziu **44 e depois 43
nódulos** — resultado diferente para entrada idêntica. Investigado até a causa
raiz, via traceback completo:

```
ValueError: autodetected range of [nan, nan] is not finite
  em radiomics/imageoperations.py, getBinEdges -> np.histogram
```

O nódulo `LIDC-IDRI-0011_N05` roda limpo quando testado isolado (5 tentativas
seguidas, sempre 61 voxels, sem erro), mas falha de forma determinística quando
processado depois de outros 10 pacientes no mesmo objeto `extractor`
reutilizado. A intensidade dentro da máscara chega como NaN somente nessa
condição — evidência de **estado acumulado no extractor entre chamadas**, não
de aleatoriedade nem de problema no dado em si.

Esta versão corrige isso e adiciona quatro camadas de proteção:

1. **Extractor novo a cada nódulo** — elimina qualquer contaminação de estado.
2. **Checagem explícita de NaN/Inf** na imagem mascarada antes de extrair,
   com motivo de descarte próprio (`intensidade_nao_finita`), em vez de deixar
   virar exceção genérica.
3. **Retry com objeto novo**: se ainda assim falhar, tenta mais uma vez antes
   de desistir do nódulo.
4. **Versão do PyRadiomics fixada por commit**, não por HEAD da branch — o
   build anterior era `3.1.1.dev111+g8ed579383`; sem fixar o hash, a próxima
   instalação do grupo pode vir de um commit diferente e não ser mais
   comparável a esta execução.
5. **Teste de determinismo embutido (Seção 13)**: roda uma amostra duas vezes
   e compara — é o teste que teria detectado o bug da v1 antes de qualquer
   análise ser feita em cima dos números.

### Ordem de execução — continua rígida

```
1 instalar -> 2 compat -> 3 Drive -> 4 parâmetros
-> 5 obter dados -> 6 escrever .pylidcrc -> 7 IMPORTAR pylidc -> 8 extrair
```

O `pylidc` lê o `.pylidcrc` **no momento do import**. Se isso acontecer antes do
caminho estar certo, todos os exames falham com `RuntimeError: Could not
establish path`, mesmo com os arquivos em disco. A Célula 7 remove o módulo da
memória antes de reimportar, então não é preciso reiniciar o ambiente se a
ordem tiver sido quebrada sem querer.

### Se a sessão cair

`/content` é efêmero. Rode de novo a partir da Célula 2. Os `.tar` no Drive
sobrevivem; o DICOM descompactado, não — mas a Célula 5 detecta isso e
descompacta de novo automaticamente, sem re-baixar do IDC.

### Estado

**Nenhum resultado deste notebook foi obtido nesta versão até você executá-lo.**
Os números da v1 (44/43 nódulos, 30 descartes por poucos leitores, 1 falha de
NaN) foram obtidos com a versão anterior do código e não valem para esta.

## 1. Instalação

Um pacote por comando: `pip` com vários pacotes aborta por inteiro quando um
falha, sem mensagem clara. PyRadiomics vem do repositório porque o sdist do
PyPI não inclui os cabeçalhos C `cmatrices.h` e `cshape.h`.

**Ordem importa.** O PyRadiomics é compilado do fonte contra o NumPy que
estiver instalado no instante do build. Se viesse primeiro, `pylidc`,
`idc-index` ou `pandas` poderiam resolver outro NumPy logo depois e trocá-lo
sob o binário já compilado. Por isso as dependências gerais são instaladas
antes e o PyRadiomics por último.

**Commit fixado**, não HEAD da branch — reprodutibilidade entre execuções e
entre integrantes do grupo depende disso. O commit abaixo é o mesmo que gerou
os resultados da v1 (`8ed579383`); troque apenas com decisão registrada.

In [1]:
PYRADIOMICS_COMMIT = '8ed579383'  # fixado deliberadamente - nao usar HEAD

# Ordem deliberada. O PyRadiomics compila as extensoes C (cmatrices, cshape)
# contra o NumPy presente no momento do build. Instalado primeiro, ele seria
# compilado contra um NumPy que as instalacoes seguintes ainda poderiam
# substituir - binario e runtime deixariam de casar, silenciosamente. Por
# isso as dependencias gerais vem antes e ele fica por ultimo.
!pip install pylidc
!pip install SimpleITK
!pip install idc-index
!pip install pandas pyarrow

# PyRadiomics por ULTIMO, ja contra o NumPy final do ambiente.
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git@{PYRADIOMICS_COMMIT}

  Cloning https://github.com/AIM-Harvard/pyradiomics.git (to revision 8ed579383) to /tmp/pip-req-build-asfpmi4j
  Running command git clone --filter=blob:none --quiet https://github.com/AIM-Harvard/pyradiomics.git /tmp/pip-req-build-asfpmi4j
  Running command git checkout -q 8ed579383
  Resolved https://github.com/AIM-Harvard/pyradiomics.git to commit 8ed579383
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.0 MB/s eta 0:00:00
  Created wheel for pyradiomics: filename=pyradiomics-3.1.1.dev111+g8ed579383-cp313-cp313-linux_x86_64.whl size=121735 sha256=fc8cde51e2e485a59eb55f8efa087a3e2128a689f42384d0adeb9d6eaac4bc73
  Stored in directory: /tmp/pip-ephem-wheel-cache-gm__lk02/wheels/

## 2. Camada de compatibilidade

Restaura o que Python 3.13 e NumPy 2.x removeram e o pylidc 0.2.3 ainda usa.
**Roda a cada reinício** — vive em memória, não é instalação.

In [2]:
compat_src = '''
"""compat.py -- restaura APIs removidas, exigidas pelo pylidc 0.2.3.

(1) configparser.SafeConfigParser: removido no Python 3.12+; usado em
    pylidc/Scan.py:55, na leitura do .pylidcrc. Como isso ocorre dentro de
    scan.to_volume(), o erro APARENTA ser falta de DICOM.
(2) np.float / np.int / np.bool: removidos no NumPy 2.x; usados em
    Contour.py:111, utils.py:144 e Annotation.py (644, 755, 1024, 1055, 1075).
"""
import configparser
import numpy as np

if not hasattr(configparser, "SafeConfigParser"):
    configparser.SafeConfigParser = configparser.ConfigParser

for _n, _t in {"float": float, "int": int, "bool": bool,
               "object": object, "str": str, "complex": complex}.items():
    if not hasattr(np, _n):
        setattr(np, _n, _t)

if not hasattr(np, "in1d"):     np.in1d = np.isin
if not hasattr(np, "alltrue"):  np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
'''

with open('/content/compat.py', 'w') as fh:
    fh.write(compat_src)

import sys
sys.path.insert(0, '/content')
import compat  # noqa: F401

import numpy as np, configparser
print('SafeConfigParser:', hasattr(configparser, 'SafeConfigParser'))
print('np.float/int/bool:', hasattr(np, 'float'), hasattr(np, 'int'), hasattr(np, 'bool'))


# --- Registro do ambiente desta sessao ---------------------------------
# As versoes sao LIDAS, nunca fixadas. SimpleITK, pandas, pyarrow e pylidc
# nao foram registrados nas rodadas historicas, entao nao existe valor de
# referencia que justifique um pin arbitrario: o que importa e poder saber,
# depois, com o que cada tabela foi produzida. O unico requisito obrigatorio
# continua sendo o PyRadiomics no commit fixado, conferido no fim da celula.
#
# pylidc e consultado pelos METADADOS da distribuicao, nunca por import: o
# modulo le o .pylidcrc no momento em que e importado, e importa-lo aqui,
# antes de o arquivo existir, quebraria a leitura dos volumes DICOM.
from importlib.metadata import PackageNotFoundError, version as _versao_dist

PACOTES_AMBIENTE = ['python', 'numpy', 'pyradiomics', 'simpleitk', 'pandas',
                    'pyarrow', 'pylidc']


def _dist(nome):
    try:
        return _versao_dist(nome)
    except PackageNotFoundError:
        return None


def _runtime_id():
    """Identificador da VM desta sessao.

    Dois notebooks executados no mesmo runtime Colab leem o mesmo boot_id. E
    o que permite comprovar, depois, que a v2 e a v3 do experimento
    definitivo sairam da MESMA sessao, sem reinicio nem reinstalacao.
    """
    try:
        with open('/proc/sys/kernel/random/boot_id') as fh:
            return fh.read().strip()
    except OSError:
        return None


AMBIENTE = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pyradiomics': _dist('pyradiomics'),
    'simpleitk': _dist('SimpleITK'),
    'pandas': _dist('pandas'),
    'pyarrow': _dist('pyarrow'),
    'pylidc': _dist('pylidc'),
    'runtime_id': _runtime_id(),
}

print('--- ambiente desta sessao ---')
for _k in PACOTES_AMBIENTE:
    print(f'  {_k:<12} {AMBIENTE[_k]}')
print(f"  {'runtime_id':<12} {AMBIENTE['runtime_id']}")

_ausentes = [k for k in PACOTES_AMBIENTE if AMBIENTE[k] is None]
if _ausentes:
    print('  AVISO: versao nao resolvida para', _ausentes)

# Requisito duro: o PyRadiomics tem de ser o build do commit fixado. Um
# release do PyPI produziria features nao comparaveis com as rodadas
# anteriores, e o engano so apareceria la na frente, no controle de Shape.
if PYRADIOMICS_COMMIT not in (AMBIENTE['pyradiomics'] or ''):
    raise RuntimeError(
        f"PyRadiomics resolvido = {AMBIENTE['pyradiomics']!r}, que nao contem "
        f'o commit fixado {PYRADIOMICS_COMMIT!r} (esperado algo como '
        f"'3.1.1.dev111+g{PYRADIOMICS_COMMIT}'). A instalacao a partir do "
        'repositorio falhou e provavelmente caiu num release do PyPI. As '
        'features nao seriam comparaveis com as rodadas anteriores. '
        'Reexecute a celula de instalacao.')

SafeConfigParser: True
np.float/int/bool: True True True
NumPy 2.1.3 | Python 3.13.15


/content/compat.py:18: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _n):
/content/compat.py:18: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _n):


## 3. Drive e caminhos

In [3]:
import os, glob, json, shutil, tarfile, time
import pandas as pd
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/PI3_Grupo4'
ARQUIVOS = f'{BASE}/dicom_tar'
FEATURES = f'{BASE}/features'
CONFIG   = f'{BASE}/config'
LOGS     = f'{BASE}/logs'
for d in (BASE, ARQUIVOS, FEATURES, CONFIG, LOGS):
    os.makedirs(d, exist_ok=True)

LIDC_ROOT = '/content/lidc'          # DICOM descompactado -- LOCAL, nunca no Drive
os.makedirs(LIDC_ROOT, exist_ok=True)

!df -h /content | tail -1

Mounted at /content/drive
overlay         108G   21G   88G  20% /


## 4. Parâmetros — decisões pendentes do grupo

**Nenhum destes valores foi decidido coletivamente.** São provisórios, para o
piloto rodar.

A tabela final guarda os **escores brutos de cada radiologista**. Consequência
prática:

| trocar | custo |
|---|---|
| `regra_rotulo`, `mediana_max_benigno`, `mediana_min_maligno` | recalcular coluna |
| `mascara`, `espacamento`, discretização | **reextrair tudo** |

### Achado da v1 que pesa nesta decisão

Na rodada piloto anterior, **38,6% dos nódulos tiveram valor central igual a
3** (17 de 44). Não é ruído de amostra pequena — é estrutural do LIDC-IDRI.
Qualquer regra de tratamento escolhida vai definir o destino de quase 4 em
cada 10 nódulos, não de um resíduo marginal. Levar esse percentual para a
discussão do grupo.

### Discretização — decisão com consequência bibliográfica

- `'binCount'` = número fixo de compartimentos (config provisória do grupo, 64).
- `'binWidth'` = largura fixa em HU. van Timmeren et al. (2020) recomendam
  largura fixa para escalas físicas como a TC; Haarburger et al. (2020) usaram
  25 HU no LIDC. Adotar `binWidth=25` torna o Experimento A comparável àquele
  trabalho.

In [4]:
PARAMS = {
    'versao_config': 'piloto_v2',

    # pré-processamento (mudar exige reextrair)
    'espacamento': [1.0, 1.0, 1.0],
    'interpolador': 'sitkBSpline',        # ver Seção 12.7 se aparecer NaN persistente
    'modo_discretizacao': 'binCount',     # 'binCount' | 'binWidth'
    'bin_count': 64,
    'bin_width': 25,

    # máscara (mudar exige reextrair)
    'mascara': 'consenso50',              # consenso50|uniao|intersecao|leitor0..3

    # seleção de nódulos
    'min_anotadores': 3,
    'diametro_min_mm': 3.0,

    # rótulo (recalculável sem reextrair)
    'regra_rotulo': 'mediana',            # mediana|media|moda
    'mediana_max_benigno': 2.0,           # protocolo Sec. 5.2: mediana <= 2.0 -> alvo 0
    'mediana_min_maligno': 4.0,           # protocolo Sec. 5.2: mediana >= 4.0 -> alvo 1

    # piloto
    'n_pacientes_piloto': 25,
    'arquivar_tar_no_drive': True,

    # robustez de extração (novo na v2)
    'max_tentativas_extracao': 2,
    'margem_contexto_voxels': 4,          # contexto para a reamostragem B-spline (Secao 9)
}

CLASSES = ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm']

for k, v in PARAMS.items():
    print(f'{k:32s} {v}')

versao_config                    piloto_v2
espacamento                      [1.0, 1.0, 1.0]
interpolador                     sitkBSpline
modo_discretizacao               binCount
bin_count                        64
bin_width                        25
mascara                          consenso50
min_anotadores                   3
diametro_min_mm                  3.0
regra_rotulo                     mediana
corte                            3
tratamento_valor_central_3       marcar
n_pacientes_piloto               25
arquivar_tar_no_drive            True
max_tentativas_extracao          2


## 5. Obter os dados

### 5.1 Diagnóstico — o que já existe?

In [5]:
def inspecionar():
    print('--- .tar no Drive ---')
    tars = sorted(glob.glob(f'{ARQUIVOS}/*.tar'))
    print(f'{ARQUIVOS}: {len(tars)} tars')

    print('\n--- DICOM já descompactado (local) ---')
    dirs = [d for d in os.listdir(LIDC_ROOT)
            if os.path.isdir(os.path.join(LIDC_ROOT, d))] if os.path.exists(LIDC_ROOT) else []
    n_dcm = len(glob.glob(f'{LIDC_ROOT}/**/*.dcm', recursive=True))
    print(f'{LIDC_ROOT}: {len(dirs)} pastas, {n_dcm} arquivos .dcm')
    return tars, dirs, n_dcm

tars, dirs_existentes, n_dcm = inspecionar()

N = PARAMS['n_pacientes_piloto']
if n_dcm > 0 and len(dirs_existentes) >= N:
    FONTE = 'ja_pronto'
elif tars:
    FONTE = 'tar_drive'
else:
    FONTE = 'idc'
print(f'\n>>> fonte: {FONTE}')

--- .tar no Drive ---
/content/drive/MyDrive/PI3_Grupo4/dicom_tar: 25 tars

--- DICOM já descompactado (local) ---
/content/lidc: 0 pastas, 0 arquivos .dcm

>>> fonte: tar_drive


### 5.2 Obter

Sem cadastro, sem manifesto `.tcia`, sem NBIA Data Retriever — direto do NCI
Imaging Data Commons.

In [6]:
if FONTE == 'ja_pronto':
    print(f'DICOM já em {LIDC_ROOT}; nada a fazer.')

elif FONTE == 'tar_drive':
    shutil.rmtree(LIDC_ROOT, ignore_errors=True); os.makedirs(LIDC_ROOT, exist_ok=True)
    for t in tars[:N]:
        with tarfile.open(t) as tf:
            tf.extractall(LIDC_ROOT)
    print(f'{min(N, len(tars))} tars descompactados')

elif FONTE == 'idc':
    from idc_index import IDCClient
    client = IDCClient.client()

    inv = client.sql_query("""
        SELECT PatientID, SeriesInstanceUID, Modality, series_size_MB
        FROM index
        WHERE collection_id = 'lidc_idri' AND Modality = 'CT'
    """)
    print(f'inventário: {inv.PatientID.nunique()} pacientes, '
          f'{inv.series_size_MB.sum()/1024:.1f} GB no total')
    inv.to_csv(f'{LOGS}/inventario_series.csv', index=False)

    alvo = sorted(inv.PatientID.unique())[:N]
    print(f'baixando {len(alvo)} pacientes: {alvo[0]} .. {alvo[-1]}')

    shutil.rmtree(LIDC_ROOT, ignore_errors=True); os.makedirs(LIDC_ROOT, exist_ok=True)
    client.download_from_selection(
        patientId=alvo, downloadDir=LIDC_ROOT,
        dirTemplate='%PatientID/%StudyInstanceUID/%SeriesInstanceUID',
    )

    if PARAMS['arquivar_tar_no_drive']:
        for pid in alvo:
            org = os.path.join(LIDC_ROOT, pid)
            if not os.path.isdir(org):
                continue
            dst = f'{ARQUIVOS}/{pid}.tar'
            if os.path.exists(dst):
                continue
            tmp = f'/content/{pid}.tar'
            with tarfile.open(tmp, 'w') as tf:
                tf.add(org, arcname=pid)
            shutil.move(tmp, dst)
        print(f'{len(glob.glob(f"{ARQUIVOS}/*.tar"))} tars no Drive')

pids_disco = sorted(d for d in os.listdir(LIDC_ROOT)
                    if os.path.isdir(os.path.join(LIDC_ROOT, d)))
n_dcm = len(glob.glob(f'{LIDC_ROOT}/**/*.dcm', recursive=True))
print(f'\n{len(pids_disco)} pacientes | {n_dcm} arquivos .dcm')

if n_dcm == 0:
    raise RuntimeError('Nenhum .dcm em disco. Verifique a Célula 5.1.')
!du -sh "$LIDC_ROOT"

/tmp/ipykernel_2636/1920692087.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(LIDC_ROOT)


25 tars descompactados

25 pacientes | 4874 arquivos .dcm
2.6G	/content/lidc


## 6. Escrever o .pylidcrc — antes do import

In [7]:
RC = os.path.expanduser('~/.pylidcrc')
with open(RC, 'w') as fh:
    fh.write(f'[dicom]\npath = {LIDC_ROOT}\nwarn = True\n')

print(open(RC).read())
print('caminho existe:', os.path.exists(LIDC_ROOT))
print('pastas na raiz :', len(pids_disco))

[dicom]
path = /content/lidc
warn = True

caminho existe: True
pastas na raiz : 25


## 7. Importar pylidc e verificar de verdade

`pl.query(pl.Scan).count()` retorna 1018 **mesmo sem nenhum DICOM em disco** —
a base de anotações vem embutida. A verificação que vale é `to_volume()`, que
lê os arquivos de fato.

In [8]:
for m in [m for m in list(sys.modules) if m.split('.')[0] == 'pylidc']:
    del sys.modules[m]

import pylidc as pl
from pylidc.utils import consensus

print('exames na base embutida:', pl.query(pl.Scan).count(), '(não confirma dados)')

scans = pl.query(pl.Scan).filter(pl.Scan.patient_id.in_(pids_disco)).all()
print(f'exames dos pacientes em disco: {len(scans)}')

s = scans[0]
v = s.to_volume()
print(f'{s.patient_id}: volume {v.shape} | pixel {s.pixel_spacing:.4f} mm | '
      f'slice {s.slice_spacing:.4f} mm | nódulos {len(s.cluster_annotations())}')
print('\nOK — leitura de DICOM funcionando.')

exames na base embutida: 1018 (não confirma dados)
exames dos pacientes em disco: 25
Loading dicom files ... This may take a moment.
LIDC-IDRI-0001: volume (512, 512, 133) | pixel 0.7031 mm | slice 2.5000 mm | nódulos 1

OK — leitura de DICOM funcionando.


## 8. Configuração do PyRadiomics

Gravada em JSON no Drive: artefato de evidência da sprint. Inclui agora o
commit fixado do PyRadiomics, não apenas a versão de desenvolvimento resolvida.

In [9]:
import logging, radiomics
from radiomics import featureextractor
radiomics.logger.setLevel(logging.ERROR)

CFG = {'resampledPixelSpacing': PARAMS['espacamento'],
       'interpolator': PARAMS['interpolador'],
       'label': 1}
if PARAMS['modo_discretizacao'] == 'binCount':
    CFG['binCount'] = PARAMS['bin_count']
elif PARAMS['modo_discretizacao'] == 'binWidth':
    CFG['binWidth'] = PARAMS['bin_width']
else:
    raise ValueError("modo_discretizacao deve ser 'binCount' ou 'binWidth'")

# instância de checagem -- apenas para confirmar que a config é válida;
# a extração de verdade cria uma instância NOVA por nódulo (Célula 9)
_checagem = featureextractor.RadiomicsFeatureExtractor(**CFG)
_checagem.disableAllFeatures()
for c in CLASSES:
    _checagem.enableFeatureClassByName(c)
print('configuração válida, classes habilitadas:', CLASSES)
del _checagem

MANIFESTO = {'params_grupo': PARAMS, 'config_pyradiomics': CFG,
             'classes_habilitadas': CLASSES,
             'pyradiomics_commit_fixado': PYRADIOMICS_COMMIT,
             'versao_pyradiomics_resolvida': radiomics.__version__,
             'versao_numpy': np.__version__,
             'versao_python': sys.version.split()[0],
             'ambiente': AMBIENTE,
             'gerado_em': time.strftime('%Y-%m-%d %H:%M:%S')}

with open(f"{CONFIG}/config_{PARAMS['versao_config']}.json", 'w') as fh:
    json.dump(MANIFESTO, fh, indent=2, ensure_ascii=False)

print(json.dumps(MANIFESTO, indent=2, ensure_ascii=False))

configuração válida, classes habilitadas: ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm']
{
  "params_grupo": {
    "versao_config": "piloto_v2",
    "espacamento": [
      1.0,
      1.0,
      1.0
    ],
    "interpolador": "sitkBSpline",
    "modo_discretizacao": "binCount",
    "bin_count": 64,
    "bin_width": 25,
    "mascara": "consenso50",
    "min_anotadores": 3,
    "diametro_min_mm": 3.0,
    "regra_rotulo": "mediana",
    "corte": 3,
    "tratamento_valor_central_3": "marcar",
    "n_pacientes_piloto": 25,
    "arquivar_tar_no_drive": true,
    "max_tentativas_extracao": 2
  },
  "config_pyradiomics": {
    "resampledPixelSpacing": [
      1.0,
      1.0,
      1.0
    ],
    "interpolator": "sitkBSpline",
    "label": 1,
    "binCount": 64
  },
  "classes_habilitadas": [
    "shape",
    "firstorder",
    "glcm",
    "glrlm",
    "glszm"
  ],
  "pyradiomics_commit_fixado": "8ed579383",
  "versao_pyradiomics_resolvida": "3.1.1.dev111+g8ed579383",
  "versao_numpy": "2.1.3"

## 9. Máscara, checagem de sanidade e extração isolada

### O que muda aqui em relação à v1

1. `extrair_features_isolado` cria um `RadiomicsFeatureExtractor` **novo a cada
   chamada**, em vez de reutilizar um objeto global. Isso é o que elimina o
   bug de contaminação de estado encontrado na v1.
2. Antes de extrair, o array de intensidade dentro da máscara é checado por
   NaN/Inf. Se houver, o nódulo é descartado com motivo específico
   (`intensidade_nao_finita`) em vez de estourar exceção genérica.
3. Se mesmo assim a extração falhar, há uma segunda tentativa com objeto
   totalmente novo antes de desistir do nódulo.

### Estratégias de máscara

| `mascara` | `clevel` | efeito |
|---|---|---|
| `consenso50` | 0,5 | voxel entra se >= 50% dos leitores marcaram |
| `uniao` | 0,01 | voxel entra se qualquer leitor marcou |
| `intersecao` | 1,0 | voxel entra se todos marcaram (pode ficar vazia) |
| `leitorN` | — | apenas a anotação de índice N |

In [10]:
import SimpleITK as sitk

CLEVEL = {'consenso50': 0.5, 'uniao': 0.01, 'intersecao': 1.0}


def montar_mascara(anns, modo):
    if modo in CLEVEL:
        cmask, cbbox, _ = consensus(anns, clevel=CLEVEL[modo])
        return cmask, cbbox
    if modo.startswith('leitor'):
        i = int(modo.replace('leitor', ''))
        if i >= len(anns):
            return None, None
        cmask, cbbox, _ = consensus([anns[i]], clevel=0.5)
        return cmask, cbbox
    raise ValueError(f'máscara desconhecida: {modo}')


def rotular(escores, p):
    # retorna (valor_central, alvo, exclusion_reason)
    if p['regra_rotulo'] == 'mediana':
        c = float(np.median(escores))
    elif p['regra_rotulo'] == 'media':
        c = float(np.mean(escores))
    elif p['regra_rotulo'] == 'moda':
        c = float(Counter(escores).most_common(1)[0][0])
    else:
        raise ValueError(p['regra_rotulo'])

    # Protocolo oficial (docs/protocolo_coorte_target_sprint2.md, Secao 5.2 e 5.3):
    #   mediana <= 2.0            -> alvo 0,    'included'
    #   mediana >= 4.0            -> alvo 1,    'included'
    #   mediana == 3.0            -> alvo None, 'consensus_indeterminate'
    #   mediana 2.5 ou 3.5        -> alvo None, 'fractional_median_even_raters'
    if c <= p['mediana_max_benigno']:
        return c, 0, 'included'
    if c >= p['mediana_min_maligno']:
        return c, 1, 'included'
    if abs(c - 3.0) < 1e-9:
        return c, None, 'consensus_indeterminate'
    return c, None, 'fractional_median_even_raters'


def extrair_features_isolado(img, msk, cfg, classes):
    """Cria um extractor NOVO por chamada. Elimina contaminação de estado
    entre nódulos, causa raiz confirmada do erro 'range of [nan, nan]'
    observado na v1 quando o extractor era reutilizado globalmente."""
    ext = featureextractor.RadiomicsFeatureExtractor(**cfg)
    ext.disableAllFeatures()
    for c in classes:
        ext.enableFeatureClassByName(c)
    return ext.execute(img, msk)


def extrair_com_retry(img, msk, cfg, classes, tentativas):
    ultimo_erro = None
    for _ in range(tentativas):
        try:
            return extrair_features_isolado(img, msk, cfg, classes)
        except Exception as e:
            ultimo_erro = e
    raise ultimo_erro


def expandir_bbox(cbbox, vol_shape, margem):
    """Expande a caixa envolvente do nodulo antes do recorte.

    Sem margem, a interpolacao B-spline (ordem 3) usada na reamostragem para
    1x1x1 mm nao tem vizinhanca suficiente nas bordas do recorte e produz
    overshoot numerico intermitente. Esta correcao vivia apenas na Secao 13.1,
    DEPOIS da extracao completa; agora faz parte da unica implementacao de
    extrair_scan, garantindo que os artefatos publicados saiam dela.
    """
    novo = []
    for eixo, tam in zip(cbbox, vol_shape):
        ini = max(0, eixo.start - margem)
        fim = min(tam, eixo.stop + margem)
        novo.append(slice(ini, fim))
    return tuple(novo)


def extrair_scan(scan, p, cfg, classes):
    vol = scan.to_volume()
    esp = [float(scan.pixel_spacing), float(scan.pixel_spacing),
           float(scan.slice_spacing)]

    linhas, descartes = [], []
    for idx, anns in enumerate(scan.cluster_annotations()):
        nid = f'{scan.patient_id}_N{idx:02d}'

        if len(anns) < p['min_anotadores']:
            descartes.append((nid, 'leitores_insuficientes')); continue

        diams = [float(a.diameter) for a in anns]
        if np.mean(diams) < p['diametro_min_mm']:
            descartes.append((nid, 'diametro_abaixo_do_minimo')); continue

        escores = [int(a.malignancy) for a in anns]
        central, alvo, motivo_alvo = rotular(escores, p)
        # indeterminados NAO sao descartados (protocolo, Secao 8, item 2): ficam
        # na tabela com alvo=None e exclusion_reason preenchido

        try:
            cmask, cbbox = montar_mascara(anns, p['mascara'])
            if cmask is None or cmask.sum() == 0:
                descartes.append((nid, 'mascara_vazia')); continue

            # --- margem de contexto antes do recorte (correcao 13.1) ---
            cbbox_exp = expandir_bbox(cbbox, vol.shape, p['margem_contexto_voxels'])
            sub = vol[cbbox_exp]

            mask_exp = np.zeros(sub.shape, dtype=cmask.dtype)
            offset = tuple(a.start - b.start for a, b in zip(cbbox, cbbox_exp))
            slices_orig = tuple(slice(o, o + s) for o, s in zip(offset, cmask.shape))
            mask_exp[slices_orig] = cmask

            # --- checagens de sanidade antes de gastar tempo no extractor ---
            valores_na_mascara = sub[mask_exp.astype(bool)]
            if not np.all(np.isfinite(valores_na_mascara)):
                descartes.append((nid, 'intensidade_nao_finita')); continue
            if np.any(np.abs(valores_na_mascara) > 5000):
                descartes.append((nid, 'intensidade_fora_de_faixa_hu')); continue

            img = sitk.GetImageFromArray(np.transpose(sub, (2, 0, 1)).astype(np.float32))
            msk = sitk.GetImageFromArray(np.transpose(mask_exp.astype(np.uint8), (2, 0, 1)))
            img.SetSpacing(esp); msk.SetSpacing(esp)

            feats = extrair_com_retry(img, msk, cfg, classes, p['max_tentativas_extracao'])
        except Exception as e:
            descartes.append((nid, f'erro_extracao:{type(e).__name__}')); continue

        linha = {'nodule_id': nid, 'patient_id': scan.patient_id,
                 'scan_id': int(scan.id), 'nodule_idx': idx,
                 'n_anotadores': len(anns),
                 'malignancy_escores': '|'.join(map(str, escores)),
                 'malignancy_mediana': float(np.median(escores)),
                 'malignancy_media': float(np.mean(escores)),
                 'malignancy_moda': float(Counter(escores).most_common(1)[0][0]),
                 'diametro_medio_mm': float(np.mean(diams)),
                 'voxels_mascara': int(cmask.sum()),
                 'valor_central': central, 'alvo': alvo,
                 'indeterminado': alvo is None,
                 'exclusion_reason': motivo_alvo,
                 'config': p['versao_config'],
                 'mascara': p['mascara']}
        linha.update({k: v for k, v in feats.items()
                      if not k.startswith('diagnostics')})
        linhas.append(linha)

    return linhas, descartes


print('funcoes definidas - implementacao UNICA de extrair_scan: extractor isolado por '
      'chamada, margem de contexto, checagem de nao-finitos e de faixa HU, e retry')

funções definidas — extractor isolado por chamada, com checagem de NaN e retry


## 10. Execução do piloto

In [38]:
todas_linhas, todos_descartes, falhas = [], [], []
t0 = time.time()

for k, pid in enumerate(pids_disco, 1):
    for scan in pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).all():
        try:
            L, D = extrair_scan(scan, PARAMS, CFG, CLASSES)
            todas_linhas.extend(L); todos_descartes.extend(D)
        except Exception as e:
            falhas.append((pid, f'{type(e).__name__}: {e}'))
    if k % 5 == 0 or k == len(pids_disco):
        print(f'{k}/{len(pids_disco)} pacientes | {len(todas_linhas)} nódulos | '
              f'{(time.time()-t0)/k:.1f}s/paciente')

print(f'\nextraídos  : {len(todas_linhas)}')
print(f'descartados: {len(todos_descartes)}')
print(f'falhas     : {len(falhas)}')

if todos_descartes:
    print('\nmotivos de descarte:')
    for m, n in Counter(d[1] for d in todos_descartes).most_common():
        print(f'  {m}: {n}')
for f in falhas[:5]:
    print('  falha:', f)

if not todas_linhas:
    raise RuntimeError(
        'Nenhum nódulo extraído. Leia os motivos acima:\n'
        '  leitores_insuficientes dominante  -> reduza min_anotadores\n'
        '  diametro_abaixo_do_minimo         -> reduza diametro_min_mm\n'
        '  mascara_vazia com intersecao      -> troque a estratégia de máscara\n'
        '  intensidade_nao_finita            -> ver Seção 12.7\n'
        '  erro_extracao:AttributeError      -> a Célula 2 não rodou\n'
        '  falhas com RuntimeError de path   -> volte às Células 5 a 7')

Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
5/25 pacientes | 7 nódulos | 2.3s/paciente
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
10/25 pacientes | 12 nódulos | 2.2s/paciente
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
15/25 pacientes | 28 nódulos | 2.1s/paciente
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loa

## 11. Tabela derivada

Identificadores e alvo primeiro, features depois. **Nenhum DICOM é versionado.**

In [12]:
df = pd.DataFrame(todas_linhas)

META = ['nodule_id', 'patient_id', 'scan_id', 'nodule_idx', 'n_anotadores',
        'malignancy_escores', 'malignancy_mediana', 'malignancy_media',
        'malignancy_moda', 'diametro_medio_mm', 'voxels_mascara',
        'valor_central', 'alvo', 'indeterminado', 'exclusion_reason',
        'config', 'mascara']
FEAT = [c for c in df.columns if c not in META]
df = df[META + sorted(FEAT)]

for c in FEAT:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')

sufixo = f"{PARAMS['versao_config']}_{PARAMS['mascara']}"
df.to_csv(f'{FEATURES}/piloto_{sufixo}.csv', index=False)
df.to_parquet(f'{FEATURES}/piloto_{sufixo}.parquet', index=False)
if todos_descartes:
    pd.DataFrame(todos_descartes, columns=['nodule_id', 'motivo']).to_csv(
        f'{FEATURES}/descartes_{sufixo}.csv', index=False)

print(f'{df.shape[0]} linhas x {df.shape[1]} colunas '
      f'({len(META)} metadados, {len(FEAT)} features)')
df.head(3)

44 linhas x 104 colunas (16 metadados, 88 features)


,nodule_id,patient_id,scan_id,nodule_idx,n_anotadores,malignancy_escores,malignancy_mediana,malignancy_media,malignancy_moda,diametro_medio_mm,...,original_shape_Maximum2DDiameterColumn,original_shape_Maximum2DDiameterRow,original_shape_Maximum2DDiameterSlice,original_shape_Maximum3DDiameter,original_shape_MeshVolume,original_shape_MinorAxisLength,original_shape_Sphericity,original_shape_SurfaceArea,original_shape_SurfaceVolumeRatio,original_shape_VoxelVolume
0,LIDC-IDRI-0001_N00,LIDC-IDRI-0001,12,0,4,5|5|5|4,5.0,4.75,5.0,32.755812,...,27.313001,31.256999,31.764760,34.914181,6738.625000,22.300427,0.695683,2480.010793,0.368029,6776.0
1,LIDC-IDRI-0003_N01,LIDC-IDRI-0003,14,1,4,5|5|3|4,4.5,4.25,5.0,31.001964,...,24.698178,26.925824,31.016125,31.032241,5316.416667,20.119504,0.748980,1966.799420,0.369948,5349.0
2,LIDC-IDRI-0003_N02,LIDC-IDRI-0003,14,2,4,4|4|3|2,3.5,3.25,4.0,13.309155,...,12.806248,11.180340,12.369317,13.152946,433.625000,8.219801,0.785666,352.632443,0.813220,445.0


## 12. Validação

Quarto item do checklist, mais uma subseção nova (12.7) específica para o
problema encontrado na v1.

In [13]:
print('=== 12.1 CONTAGEM POR CLASSE ===')
por_classe = {}
for c in CLASSES:
    cols = [x for x in FEAT if f'_{c}_' in x]
    por_classe[c] = len(cols)
    print(f'  {c:12s} {len(cols):3d}')
print(f'  {"TOTAL":12s} {sum(por_classe.values()):3d}')

orfas = [x for x in FEAT if not any(f'_{c}_' in x for c in CLASSES)]
if orfas:
    print(f'  ATENÇÃO: {len(orfas)} colunas fora das 5 classes:', orfas[:5])
faltando = [c for c, n in por_classe.items() if n == 0]
print('  classes ausentes:', faltando if faltando else 'nenhuma')

print('\n=== 12.2 TIPOS ===')
nao_num = [c for c in FEAT if not pd.api.types.is_numeric_dtype(df[c])]
print(f'  não numéricas: {len(nao_num)}', nao_num[:5] if nao_num else '')
print(f'  dtypes: {set(str(df[c].dtype) for c in FEAT)}')

print('\n=== 12.3 AUSENTES E DEGENERADAS ===')
na = df[FEAT].isna().sum(); na = na[na > 0].sort_values(ascending=False)
print(f'  colunas com NaN: {len(na)}')
if len(na):
    print(na.head(10).to_string())
inf = {c: int(np.isinf(df[c]).sum()) for c in FEAT if np.isinf(df[c]).any()}
print(f'  colunas com inf: {len(inf)}', list(inf.items())[:5] if inf else '')
const = [c for c in FEAT if df[c].nunique(dropna=True) <= 1]
print(f'  colunas constantes: {len(const)}', const[:5] if const else '')
lin_na = int(df[FEAT].isna().any(axis=1).sum())
print(f'  linhas com algum NaN: {lin_na} de {len(df)}')

=== 12.1 CONTAGEM POR CLASSE ===
  shape         14
  firstorder    18
  glcm          24
  glrlm         16
  glszm         16
  TOTAL         88
  classes ausentes: nenhuma

=== 12.2 TIPOS ===
  não numéricas: 0 
  dtypes: {'float64'}

=== 12.3 AUSENTES E DEGENERADAS ===
  colunas com NaN: 0
  colunas com inf: 0 
  colunas constantes: 0 
  linhas com algum NaN: 0 de 44


In [14]:
print('=== 12.4 COBERTURA E ALVO ===')
print(f'  pacientes com nódulo: {df.patient_id.nunique()} de {len(pids_disco)}')
g = df.groupby('patient_id').size()
print(f'  nódulos por paciente: mediana {g.median():.1f}, máx {g.max()}')

print(f"\n  valor central (regra = {PARAMS['regra_rotulo']}):")
print(df.valor_central.value_counts().sort_index().to_string())
n3 = int(df.indeterminado.sum())
pct3 = 100*n3/len(df) if len(df) else 0
print(f'\n  indeterminados (mediana 2.5/3.0/3.5): {n3} ({pct3:.1f}%)')
print('  por exclusion_reason:')
print(df.exclusion_reason.value_counts().to_string())
print('\n  alvo binário:'); print(df.alvo.value_counts(dropna=False).to_string())
print('\n  anotadores por nódulo:')
print(df.n_anotadores.value_counts().sort_index().to_string())

print('\n=== 12.5 SANIDADE FÍSICA ===')
vc = [c for c in FEAT if c.endswith('shape_VoxelVolume')]
if vc:
    v = df[vc[0]]
    print(f'  VoxelVolume mm³: mín {v.min():.1f} | mediana {v.median():.1f} | máx {v.max():.1f}')
    print(f'  volume < 14 mm³ (esfera de 3 mm): {(v < 14).sum()}')
ec = [c for c in FEAT if c.endswith('shape_Sphericity')]
if ec:
    e = df[ec[0]]
    fora = int(((e < 0) | (e > 1)).sum())
    print(f'  Sphericity: mín {e.min():.3f} | máx {e.max():.3f} | fora de [0,1]: {fora}')
    if fora:
        print('  >>> ERRO DE GEOMETRIA: imagem e máscara em espaços diferentes')
print(f'  voxels na máscara: mín {df.voxels_mascara.min()} | '
      f'mediana {df.voxels_mascara.median():.0f} | máx {df.voxels_mascara.max()}')

=== 12.4 COBERTURA E ALVO ===
  pacientes com nódulo: 20 de 25
  nódulos por paciente: mediana 1.0, máx 9

  valor central (regra = mediana):
valor_central
1.0     3
2.0     2
2.5     3
3.0    17
3.5     9
4.0     2
4.5     3
5.0     5

  indeterminados (valor central 3): 17 (38.6%) — decisão pendente do grupo

  alvo binário:
alvo
1.0    19
NaN    17
0.0     8

  anotadores por nódulo:
n_anotadores
3     9
4    35

=== 12.5 SANIDADE FÍSICA ===
  VoxelVolume mm³: mín 55.0 | mediana 214.5 | máx 7858.0
  volume < 14 mm³ (esfera de 3 mm): 0
  Sphericity: mín 0.534 | máx 0.889 | fora de [0,1]: 0
  voxels na máscara: mín 31 | mediana 197 | máx 5428


In [15]:
print('=== 12.6 RELATÓRIO ===')
motivos = dict(Counter(d[1] for d in todos_descartes))
# rastreabilidade do protocolo, Secao 5.3: as duas causas de indefinicao de
# alvo sao contadas a partir de exclusion_reason e reportadas separadamente
n_consensus_indeterminate = int(
    (df.exclusion_reason == 'consensus_indeterminate').sum())
n_fractional_median = int(
    (df.exclusion_reason == 'fractional_median_even_raters').sum())
n_indeterminado_col = int(df['indeterminado'].sum())
assert n3 == n_consensus_indeterminate + n_fractional_median, (
    f'indeterminados ({n3}) != consensus_indeterminate '
    f'({n_consensus_indeterminate}) + fractional_median_even_raters '
    f'({n_fractional_median})')
assert n3 == n_indeterminado_col, (
    f'indeterminados ({n3}) != soma da coluna indeterminado '
    f'({n_indeterminado_col})')
relatorio = {'config': MANIFESTO,
             'pacientes_piloto': len(pids_disco),
             'pacientes_com_nodulo': int(df.patient_id.nunique()),
             'nodulos_extraidos': int(len(df)),
             'nodulos_descartados': len(todos_descartes),
             'motivos_descarte': motivos,
             'falhas': len(falhas),
             'features_por_classe': por_classe,
             'features_total': int(sum(por_classe.values())),
             'colunas_com_nan': int(len(na)), 'colunas_com_inf': len(inf),
             'colunas_constantes': len(const), 'linhas_com_nan': lin_na,
             'indeterminados': n3,
             'consensus_indeterminate': n_consensus_indeterminate,
             'fractional_median_even_raters': n_fractional_median,
             'alvo_0': int((df.alvo == 0).sum()),
             'alvo_1': int((df.alvo == 1).sum())}
with open(f'{FEATURES}/validacao_{sufixo}.json', 'w') as fh:
    json.dump(relatorio, fh, indent=2, ensure_ascii=False, default=str)

print(json.dumps({k: v for k, v in relatorio.items() if k != 'config'},
                 indent=2, ensure_ascii=False))

print('\n=== 12.7 CASOS DE INTENSIDADE NÃO FINITA (o bug da v1) ===')
n_nan = motivos.get('intensidade_nao_finita', 0)
if n_nan == 0:
    print('  nenhum caso nesta rodada — o problema encontrado na v1',
          '(extractor reutilizado entre nódulos) não se repetiu.')
else:
    ids_nan = [d[0] for d in todos_descartes if d[1] == 'intensidade_nao_finita']
    print(f'  {n_nan} nódulo(s) com intensidade não finita dentro da máscara:')
    print(' ', ids_nan)
    print('  Isolar cada um com montar_mascara() + inspeção manual do array',
          'antes de decidir se é problema de dado ou de reamostragem B-spline.')

=== 12.6 RELATÓRIO ===
{
  "pacientes_piloto": 25,
  "pacientes_com_nodulo": 20,
  "nodulos_extraidos": 44,
  "nodulos_descartados": 30,
  "motivos_descarte": {
    "leitores_insuficientes": 30
  },
  "falhas": 0,
  "features_por_classe": {
    "shape": 14,
    "firstorder": 18,
    "glcm": 24,
    "glrlm": 16,
    "glszm": 16
  },
  "features_total": 88,
  "colunas_com_nan": 0,
  "colunas_com_inf": 0,
  "colunas_constantes": 0,
  "linhas_com_nan": 0,
  "indeterminados": 17,
  "alvo_0": 8,
  "alvo_1": 19
}

=== 12.7 CASOS DE INTENSIDADE NÃO FINITA (o bug da v1) ===
  nenhum caso nesta rodada — o problema encontrado na v1 (extractor reutilizado entre nódulos) não se repetiu.


## 13. Reprodutibilidade: rodar duas vezes e comparar

Esta célula é o teste que teria pego o bug da v1 antes de qualquer análise.
Roda a extração de um subconjunto pequeno duas vezes e compara — se os números
não baterem, algo no pipeline ainda não é determinístico.

In [17]:
N_TESTE = min(10, len(pids_disco))
amostra_teste = pids_disco[:N_TESTE]


def rodar_amostra():
    linhas = []
    for pid in amostra_teste:
        for scan in pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).all():
            L, _ = extrair_scan(scan, PARAMS, CFG, CLASSES)
            linhas.extend(L)
    return pd.DataFrame(linhas).sort_values('nodule_id').reset_index(drop=True)


print(f'Rodando 2x sobre {N_TESTE} pacientes para checar determinismo...')
r1 = rodar_amostra()
r2 = rodar_amostra()

mesmas_linhas = len(r1) == len(r2)
mesmos_ids = set(r1.nodule_id) == set(r2.nodule_id)
print(f'  rodada 1: {len(r1)} nódulos | rodada 2: {len(r2)} nódulos')
print(f'  mesma contagem: {mesmas_linhas} | mesmos IDs: {mesmos_ids}')

if mesmas_linhas and mesmos_ids:
    cols_num = [c for c in r1.columns
            if pd.api.types.is_numeric_dtype(r1[c]) and r1[c].dtype != bool]
    r1s = r1.set_index('nodule_id')[cols_num]
    r2s = r2.set_index('nodule_id')[cols_num]
    diff = (r1s - r2s).abs()
    max_diff = diff.max().max()
    print(f'  maior diferença absoluta entre as duas rodadas: {max_diff:.2e}')
    print('  OK, pipeline determinístico' if max_diff < 1e-6
          else '  ATENÇÃO: ainda há não-determinismo — investigar antes de escalar')
else:
    print('  ATENÇÃO: rodadas produziram conjuntos diferentes de nódulos —',
          'investigar antes de escalar para os 1.010 pacientes')

Rodando 2x sobre 10 pacientes para checar determinismo...
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.


/usr/local/lib/python3.13/dist-packages/numpy/_core/function_base.py:146: RuntimeWarning: overflow encountered in subtract
  delta = np.subtract(stop, start, dtype=type(dt))
/usr/local/lib/python3.13/dist-packages/numpy/_core/function_base.py:169: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.13/dist-packages/numpy/lib/_histograms_impl.py:353: RuntimeWarning: overflow encountered in subtract
  return np.subtract(a, b, dtype=dt)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_histograms_impl.py:856: RuntimeWarning: invalid value encountered in divide
  f_indices = ((_unsigned_subtract(tmp_a, first_edge) / norm_denom)
/usr/local/lib/python3.13/dist-packages/numpy/lib/_histograms_impl.py:858: RuntimeWarning: invalid value encountered in cast
  indices = f_indices.astype(np.intp)


Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
  rodada 1: 11 nódulos | rodada 2: 12 nódulos
  mesma contagem: False | mesmos IDs: False
  ATENÇÃO: rodadas produziram conjuntos diferentes de nódulos — investigar antes de escalar para os 1.010 pacientes


In [18]:
for tentativa in range(3):
    grupos_por_paciente = {}
    for pid in amostra_teste:
        scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).first()
        grupos = scan.cluster_annotations()
        grupos_por_paciente[pid] = len(grupos)
    print(f'tentativa {tentativa}:', grupos_por_paciente)

tentativa 0: {'LIDC-IDRI-0001': 1, 'LIDC-IDRI-0002': 1, 'LIDC-IDRI-0003': 4, 'LIDC-IDRI-0004': 1, 'LIDC-IDRI-0005': 3, 'LIDC-IDRI-0006': 4, 'LIDC-IDRI-0007': 2, 'LIDC-IDRI-0008': 2, 'LIDC-IDRI-0009': 2, 'LIDC-IDRI-0010': 3}
tentativa 1: {'LIDC-IDRI-0001': 1, 'LIDC-IDRI-0002': 1, 'LIDC-IDRI-0003': 4, 'LIDC-IDRI-0004': 1, 'LIDC-IDRI-0005': 3, 'LIDC-IDRI-0006': 4, 'LIDC-IDRI-0007': 2, 'LIDC-IDRI-0008': 2, 'LIDC-IDRI-0009': 2, 'LIDC-IDRI-0010': 3}
tentativa 2: {'LIDC-IDRI-0001': 1, 'LIDC-IDRI-0002': 1, 'LIDC-IDRI-0003': 4, 'LIDC-IDRI-0004': 1, 'LIDC-IDRI-0005': 3, 'LIDC-IDRI-0006': 4, 'LIDC-IDRI-0007': 2, 'LIDC-IDRI-0008': 2, 'LIDC-IDRI-0009': 2, 'LIDC-IDRI-0010': 3}


## 13.1 Correção de margem de contexto — onde ela vive agora

Sem margem ao redor do nódulo, a interpolação B-spline usada na reamostragem
para 1×1×1 mm não tem vizinhança suficiente nas bordas do recorte e produz
overshoot numérico intermitente: o mesmo nódulo ora era extraído, ora
descartado entre execuções idênticas.

**Esta correção não é mais uma redefinição posterior.** `expandir_bbox`, a
expansão da caixa envolvente e a checagem de faixa de Hounsfield fazem parte
da **única** implementação de `extrair_scan`, definida na Seção 9 e usada pela
extração completa da Seção 10 — a mesma que gera o CSV, o Parquet e o JSON de
validação. A célula abaixo apenas confirma, sem reprocessar nada, que é essa
implementação que está em vigor.

Motivo da mudança estrutural: enquanto a correção era aplicada apenas aqui, os
artefatos publicados saíam da versão anterior da função, com geometria
diferente da que o notebook declarava ter corrigido.

In [ ]:
# A correcao de margem de contexto NAO e mais uma redefinicao posterior: ela faz
# parte da unica implementacao de extrair_scan (Secao 9), que e a usada pela
# extracao completa da Secao 10 - a que grava CSV, Parquet e validacao_*.json.
# Esta celula so verifica, estaticamente e sem reprocessar nada, que e isso que
# esta em vigor. Se falhar, NAO prossiga: os artefatos sairiam com a geometria
# antiga, exatamente o defeito diagnosticado no Bloqueador 2.
import inspect

src_extrair = inspect.getsource(extrair_scan)

checagens = {
    'expandir_bbox definido antes da extracao': 'expandir_bbox' in globals(),
    'extrair_scan usa expandir_bbox': 'expandir_bbox(' in src_extrair,
    'margem vem do PARAMS, nao hardcoded': "p['margem_contexto_voxels']" in src_extrair,
    'PARAMS traz margem_contexto_voxels = 4': PARAMS.get('margem_contexto_voxels') == 4,
    'MANIFESTO registra a margem': MANIFESTO['params_grupo'].get('margem_contexto_voxels') == 4,
    'checagem de intensidade nao finita': 'intensidade_nao_finita' in src_extrair,
    'checagem de faixa HU': 'intensidade_fora_de_faixa_hu' in src_extrair,
    'esta celula nao redefine extrair_scan':
        extrair_scan.__module__ == '__main__' and 'def extrair_scan' not in src_extrair[1:],
}

for rotulo, ok in checagens.items():
    print(f"  [{'OK   ' if ok else 'FALHA'}] {rotulo}")

if not all(checagens.values()):
    raise RuntimeError(
        'A extrair_scan em vigor NAO e a implementacao corrigida. Nao prossiga: '
        'a tabela de features sairia com a geometria antiga (sem margem de '
        'contexto), que foi a causa da comparacao invalida entre binCount e '
        'binWidth diagnosticada na Etapa 2.1.')

print()
print('Implementacao unica confirmada. A tabela da Secao 11 e os artefatos da '
      'Secao 12 saem desta mesma funcao.')

In [35]:
r1 = rodar_amostra()
r2 = rodar_amostra()

print(f'rodada 1: {len(r1)} nódulos | rodada 2: {len(r2)} nódulos')
print(f'mesma contagem: {len(r1) == len(r2)} | mesmos IDs: {set(r1) == set(r2) if isinstance(r1, set) else "ver formato de r1/r2"}')

Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
rodada 1: 12 nódulos | rodada 2: 12 nódu

## 13.2 Verificação de identificadores pós-correção

Contexto:

Confirma se as duas execuções da amostra de 10 pacientes extraíram exatamente os mesmos nódulos, comparando os nodule_id um a um. É a segunda parte do teste de determinismo — a primeira (contagem) já bateu em 12 e 12; esta fecha a verificação de que são os mesmos 12, e não 12 diferentes por coincidência de número.

In [36]:
# Extrai só os nodule_id de cada resultado, não importa se r1/r2 são listas de dicts
ids1 = {item['nodule_id'] for item in r1} if isinstance(r1, list) else set(r1)
ids2 = {item['nodule_id'] for item in r2} if isinstance(r2, list) else set(r2)

print('mesmos IDs:', ids1 == ids2)
print('só na execução 1:', ids1 - ids2)
print('só na execução 2:', ids2 - ids1)

mesmos IDs: True
só na execução 1: set()
só na execução 2: set()


## 13.3 Comparação numérica entre execuções — teste final de determinismo

Contexto:

Última etapa do teste: verifica se, além dos mesmos nódulos, os valores de cada feature são idênticos entre as duas execuções. É este resultado que confirma ou refuta, de forma definitiva, se a correção da Seção 13.1 eliminou o não determinismo diagnosticado nas rodadas anteriores. Diferença máxima abaixo de 1e-6 é considerada equivalente a zero, tolerância de ponto flutuante.

In [37]:
import pandas as pd

df1 = pd.DataFrame(r1).set_index('nodule_id').sort_index()
df2 = pd.DataFrame(r2).set_index('nodule_id').sort_index()

cols_num = [c for c in df1.columns
            if pd.api.types.is_numeric_dtype(df1[c]) and df1[c].dtype != bool]

diff = (df1[cols_num] - df2[cols_num]).abs()
max_diff = diff.max().max()
print(f'maior diferença absoluta entre as duas execuções: {max_diff:.2e}')
print('determinístico' if max_diff < 1e-6 else 'ATENÇÃO: ainda há diferença numérica')

maior diferença absoluta entre as duas execuções: 0.00e+00
determinístico


## 14. Saídas e como ler os resultados

```
PI3_Grupo4/
├── dicom_tar/                            <- 1 tar por paciente (se arquivado)
├── config/config_piloto_v2.json          <- evidência: parâmetros + versões
│                                             + commit fixado do PyRadiomics
├── logs/inventario_series.csv
└── features/
    ├── piloto_piloto_v2_consenso50.csv   <- tabela derivada
    ├── piloto_piloto_v2_consenso50.parquet
    ├── descartes_...csv                  <- inclui 'intensidade_nao_finita'
    └── validacao_...json
```

**Leitura da validação:**

- Classe com contagem zero: não foi habilitada ou falhou.
- `Sphericity` fora de [0,1]: erro de geometria, investigar antes de seguir.
- NaN/inf em textura: cruzar com `voxels_mascara`; se concentrar em nódulos
  pequenos, conversa com Haarburger et al. (2020) sobre baixa reprodutibilidade
  de atributos em estruturas pequenas.
- **`intensidade_nao_finita` (Seção 12.7):** se aparecer aqui mesmo com o
  extractor isolado por nódulo, o problema não era de estado compartilhado —
  é a interpolação B-spline produzindo valor inválido para aquela geometria
  específica. Nesse caso, teste `'interpolador': 'sitkLinear'` para os casos
  afetados (menos suave, mas numericamente mais estável em regiões pequenas
  ou próximas à borda da imagem).
- **Seção 13:** se a comparação de determinismo apontar diferença, não
  prossiga para a extração completa antes de entender a causa — o mesmo tipo
  de bug da v1 pode estar presente em outro ponto do código.

**O que este piloto não responde:** estabilidade dos atributos frente à
variabilidade de segmentação (ICC entre leitores), comparação entre estratégias
de máscara, e ganho sobre o baseline.

**Versionar no repositório:** este notebook, `compat.py`, `requirements.txt`,
`config_*.json` e `validacao_*.json`. A tabela de features é dado derivado — o
grupo decide se versiona. **DICOM não entra no repositório.**